In [1]:
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense
from tensorflow.keras import layers


from keras import layers
from keras import Input
from keras.models import Model

import numpy as np
import tqdm
import keras
import tensorflow as tf
import os
import csv
import pathlib
import unicode

#from tensorflow.python.keras.preprocessing.image import ImageDataGenerator

2024-03-18 16:37:47.040855: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2024-03-18 16:37:47.713561: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


In [91]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "1"

print(tf.__version__)
from tensorflow.python.client import device_lib
device_lib.list_local_devices()

2.16.1


2024-03-18 16:25:41.684534: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:984] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2024-03-18 16:25:41.684605: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:984] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2024-03-18 16:25:41.684647: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:984] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2024-03-18 16:25:41.684775: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:984] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2024-03-18 16:25:41.684785: I tensorflow/core/common_runtime/gpu/gpu

[name: "/device:CPU:0"
 device_type: "CPU"
 memory_limit: 268435456
 locality {
 }
 incarnation: 11949726540119887950
 xla_global_id: -1,
 name: "/device:GPU:0"
 device_type: "GPU"
 memory_limit: 2355888128
 locality {
   bus_id: 1
   links {
   }
 }
 incarnation: 5106804276295066852
 physical_device_desc: "device: 0, name: NVIDIA GeForce GTX 1650, pci bus id: 0000:01:00.0, compute capability: 7.5"
 xla_global_id: 416903419]

In [19]:
ja2label = {'ㄱ':0, 'ㄲ':1, 'ㄴ':2, 'ㄷ':3, 'ㄸ':4, 'ㄹ':5, 'ㅁ':6, 'ㅂ':7, 'ㅃ':8,
'ㅅ':9, 'ㅆ':10, 'ㅇ':11,  'ㅈ':12, 'ㅉ':13, 'ㅊ':14, 'ㅋ':15, 'ㅌ':16,  'ㅍ':17, 'ㅎ':18}

mo2label = {'ㅏ':0, 'ㅐ':1, 'ㅑ':2, 'ㅒ':3, 'ㅓ':4, 'ㅔ':5, 'ㅕ':6, 'ㅖ':7, 'ㅗ':8, 'ㅘ':9, 
'ㅙ':10, 'ㅚ':11, 'ㅛ':12, 'ㅜ':13, 'ㅝ':14, 'ㅞ':15, 'ㅟ':16, 'ㅠ':17, 'ㅡ':18, 'ㅢ':19, 'ㅣ':20}

ba2label = {None:0, 'ㄱ':1, 'ㄲ':2, 'ㄳ':3, 'ㄴ':4, 'ㄵ':5, 'ㄶ':6, 'ㄷ':7, 'ㄹ':8, 'ㄺ':9,
'ㄻ':10, 'ㄼ':11, 'ㄽ':12, 'ㄾ':13, 'ㄿ':14, 'ㅀ':15, 'ㅁ':16, 'ㅂ':17, 'ㅄ':18, 'ㅅ':19,
'ㅆ':20, 'ㅇ':21, 'ㅈ':22, 'ㅊ':23, 'ㅋ':24, 'ㅌ':25, 'ㅍ':26, 'ㅎ':27}

label2ja = {0: 'ㄱ', 1: 'ㄲ', 2: 'ㄴ', 3: 'ㄷ', 4: 'ㄸ', 5: 'ㄹ',
            6: 'ㅁ', 7: 'ㅂ', 8: 'ㅃ', 9: 'ㅅ', 10: 'ㅆ', 11: 'ㅇ',
            12: 'ㅈ', 13: 'ㅉ', 14: 'ㅊ', 15: 'ㅋ', 16: 'ㅌ', 17: 'ㅍ', 18: 'ㅎ'}

label2mo = {0: 'ㅏ', 1: 'ㅐ', 2: 'ㅑ', 3: 'ㅒ', 4: 'ㅓ', 5: 'ㅔ',
            6: 'ㅕ', 7: 'ㅖ', 8: 'ㅗ', 9: 'ㅘ', 10: 'ㅙ', 11: 'ㅚ',
            12: 'ㅛ', 13: 'ㅜ', 14: 'ㅝ', 15: 'ㅞ', 16: 'ㅟ', 17: 'ㅠ',
            18: 'ㅡ', 19: 'ㅢ', 20: 'ㅣ'}

label2ba = {0: None, 1: 'ㄱ', 2: 'ㄲ', 3: 'ㄳ', 4: 'ㄴ', 5: 'ㄵ',
            6: 'ㄶ', 7: 'ㄷ', 8: 'ㄹ', 9: 'ㄺ', 10: 'ㄻ', 11: 'ㄼ',
            12: 'ㄽ', 13: 'ㄾ', 14: 'ㄿ', 15: 'ㅀ', 16: 'ㅁ', 17: 'ㅂ',
            18: 'ㅄ', 19: 'ㅅ', 20: 'ㅆ', 21: 'ㅇ', 22: 'ㅈ', 23: 'ㅊ',
            24: 'ㅋ', 25: 'ㅌ', 26: 'ㅍ', 27: 'ㅎ'}

In [4]:
pip install pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.7/12.7 MB 7.5 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 505.5/505.5 kB 6.9 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 345.4/345.4 kB 6.8 MB/s eta 0:00:00a 0:00:01
Note: you may need to restart the kernel to use updated packages.


In [5]:
import pandas as pd

In [11]:
reader

,/root/Data/hangul/image/115/11530001036.jpg,0,3,8
0,/root/Data/hangul/image/115/11530018074.jpg,18,11,17
1,/root/Data/hangul/image/115/11530001123.jpg,0,13,9
2,/root/Data/hangul/image/115/11530004050.jpg,3,0,10
3,/root/Data/hangul/image/115/11530013018.jpg,12,1,8
4,/root/Data/hangul/image/115/11530017089.jpg,17,12,8
...,...,...,...,...
41489,/root/Data/hangul/image/101/10130018060.jpg,18,9,1
41490,/root/Data/hangul/image/101/10130003022.jpg,1,18,21
41491,/root/Data/hangul/image/101/10130002089.jpg,1,8,0
41492,/root/Data/hangul/image/101/10130014117.jpg,14,4,4


In [12]:
reader.head(10)

,/root/Data/hangul/image/115/11530001036.jpg,0,3,8
0,/root/Data/hangul/image/115/11530018074.jpg,18,11,17
1,/root/Data/hangul/image/115/11530001123.jpg,0,13,9
2,/root/Data/hangul/image/115/11530004050.jpg,3,0,10
3,/root/Data/hangul/image/115/11530013018.jpg,12,1,8
4,/root/Data/hangul/image/115/11530017089.jpg,17,12,8
5,/root/Data/hangul/image/115/11530006007.jpg,5,0,26
6,/root/Data/hangul/image/115/11530005128.jpg,5,0,1
7,/root/Data/hangul/image/115/11530014054.jpg,13,9,8
8,/root/Data/hangul/image/115/11530004080.jpg,3,4,17
9,/root/Data/hangul/image/115/11530011022.jpg,10,13,1


In [22]:
reader = pd.read_csv("/root/Data/hangul/dataset/test.csv", encoding='utf-8', header=None)
idx = 1000
print(reader.iloc[idx, :][0])
ja = label2ja[reader.iloc[idx, :][1]]
mo = label2mo[reader.iloc[idx, :][2]]
ba = label2ba[reader.iloc[idx, :][3]]

char = unicode.join_jamos_char(ja, mo ,ba)
print(char)


/root/Data/hangul/image/115/11530005008.jpg
뒤


In [3]:

def test(idx):
    csvFile = open("/root/Data/hangul/dataset/test.csv", 'r', encoding='utf-8')
    
    
    line = reader.k
    print(line[1])
    
test(10)

TypeError: '_csv.reader' object is not subscriptable

In [93]:
def get_dataset_fromCsv():
    cnt = 0
    print("Helelo?")
    
    csvFile = open("/root/Data/hangul/dataset/test.csv", 'r', encoding='utf-8')
    reader = csv.reader(csvFile)
    
    for line in reader:
        imgFile = line[0]
        #print(imgFile)
        img = tf.io.read_file(imgFile)
        img = tf.image.decode_jpeg(img, channels=3)
        img = tf.image.convert_image_dtype(img, tf.float32)
        img = tf.image.resize(img, (64, 64))    
        img = np.array(img)
        img = np.expand_dims(img, axis=0)
        
        label1 = np.expand_dims(np.array(int(line[1])), axis=0)
        label2 = np.expand_dims(np.array(int(line[2])), axis=0)
        label3 = np.expand_dims(np.array(int(line[3])), axis=0)

        yield img, (label1, label2, label3)

        
        
        
        # def dataset_generator():
        #     #labels = np.array([int(line[1]), int(line[2]), int(line[3])], dtype= 'int64')
        #     # print(cnt)
        #     print("rtn")
        #     label1 = np.expand_dims(np.array(int(line[1])), axis=0)
        #     label2 = np.expand_dims(np.array(int(line[2])), axis=0)
        #     label3 = np.expand_dims(np.array(int(line[3])), axis=0)

        #     yield img, (label1, label2, label3)

        #                              args=("/root/Data/hangul/dataset/tranDataset.csv",))
        

        #dataset = dataset.batch(16).take(10)
        
        

           
        
        
    
#get_dataset_fromCsv("/root/Data/hangul/dataset/tranDataset.csv")

In [94]:
dataset = tf.data.Dataset.from_generator(get_dataset_fromCsv,
                                            output_signature=
                                            (
                                            tf.TensorSpec(shape = (None, 64, 64, 3), dtype =tf.float32, name = "posts"),
                                            (tf.TensorSpec(shape = (1,), dtype = tf.int64, name = "DenseCho2"),
                                            tf.TensorSpec(shape = (1,), dtype = tf.int64, name = "DenseJung2"),
                                            tf.TensorSpec(shape = (1,), dtype = tf.int64, name = "DenseJong2"))
                                            )
                                            #output_shapes=([19,21,28])
                                            )

print(dataset)
iterator = iter(dataset)
print(next(iterator))

<_FlatMapDataset element_spec=(TensorSpec(shape=(None, 64, 64, 3), dtype=tf.float32, name='posts'), (TensorSpec(shape=(1,), dtype=tf.int64, name='DenseCho2'), TensorSpec(shape=(1,), dtype=tf.int64, name='DenseJung2'), TensorSpec(shape=(1,), dtype=tf.int64, name='DenseJong2')))>
Helelo?
(<tf.Tensor: shape=(1, 64, 64, 3), dtype=float32, numpy=
array([[[[1.       , 1.       , 1.       ],
         [1.       , 1.       , 1.       ],
         [1.       , 1.       , 1.       ],
         ...,
         [0.9960785, 0.9960785, 0.9960785],
         [0.9960785, 0.9960785, 0.9960785],
         [0.9960785, 0.9960785, 0.9960785]],

        [[1.       , 1.       , 1.       ],
         [1.       , 1.       , 1.       ],
         [1.       , 1.       , 1.       ],
         ...,
         [0.9960785, 0.9960785, 0.9960785],
         [0.9960785, 0.9960785, 0.9960785],
         [0.9960785, 0.9960785, 0.9960785]],

        [[1.       , 1.       , 1.       ],
         [1.       , 1.       , 1.       ],
        

In [95]:

posts_input = Input(shape=(64,64,3), dtype='float32', name='posts')

x = layers.Flatten()(posts_input)

DenseCho = layers.Dense(128, activation='relu', name='DenseCho1')(x)
DenseJung = layers.Dense(128, activation='relu', name='DenseJung1')(x)
DenseJong = layers.Dense(128, activation='relu', name='DenseJong1')(x)


DenseCho = layers.Dense(19, activation='softmax', name='DenseCho2')(DenseCho)
DenseJung = layers.Dense(21, activation='softmax', name='DenseJung2')(DenseJung)
DenseJong = layers.Dense(28, activation='softmax', name='DenseJong2')(DenseJong)

losses = {
	#"DenseCho2": "categorical_crossentropy",
	"DenseCho2": "sparse_categorical_crossentropy",
	"DenseJung2": "sparse_categorical_crossentropy",
    "DenseJong2": "sparse_categorical_crossentropy"
}

model = Model(posts_input, [DenseCho, DenseJung, DenseJong])

# model.compile(loss=losses, optimizer='adam', metrics=[['accuracy'], ['accuracy'], ['accuracy']])
# model.compile(loss=["sparse_categorical_crossentropy", "sparse_categorical_crossentropy", "sparse_categorical_crossentropy"], 
#               optimizer='adam', 
#               metrics=[['accuracy'], ['accuracy'], ['accuracy']])


# model.compile(loss=["categorical_crossentropy", "categorical_crossentropy", "categorical_crossentropy"], 
#               optimizer='adam', 
#               metrics=[['accuracy'], ['accuracy'], ['accuracy']])

model.compile(loss = 'sparse_categorical_crossentropy',optimizer='adam', 
               metrics=[['accuracy'], ['accuracy'], ['accuracy']])

In [97]:
#dataset = tf.data.Dataset.from_tensor_slices(({'input_x': data_a, 'input_y': data_b}, labels)).batch(2).repeat()
#model.fit(get_dataset_fromCsv("/root/Data/hangul/dataset/tranDataset.csv"), epochs = 100, batch_size = 16)
#dataset = dataset.shuffle(150).batch(8)
model.fit(dataset, epochs = 100, batch_size = 16)
#model.fit_generator(dataset, epochs = 100)


Epoch 1/100
Helelo?
41495/41495 ━━━━━━━━━━━━━━━━━━━━ 158s 4ms/step - DenseCho2_accuracy: 0.0874 - DenseJong2_accuracy: 0.1467 - DenseJung2_accuracy: 0.1000 - loss: 8.7434
Epoch 2/100
Helelo?
   43/41495 ━━━━━━━━━━━━━━━━━━━━ 2:31 4ms/step - DenseCho2_accuracy: 0.0254 - DenseJong2_accuracy: 0.0472 - DenseJung2_accuracy: 0.2252 - loss: 8.6305          

2024-03-18 16:28:47.331112: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
	 [[{{node IteratorGetNext}}]]
/root/anaconda3/envs/Anaconda_tensor/lib/python3.12/contextlib.py:158: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self.gen.throw(value)
2024-03-18 16:28:47.331170: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
	 [[{{node IteratorGetNext}}]]
	 [[IteratorGetNext/_4]]
2024-03-18 16:28:47.331201: I tensorflow/core/framework/local_rendezvous.cc:422] Local rendezvous recv item cancelled. Key hash: 14274495685733646192


41495/41495 ━━━━━━━━━━━━━━━━━━━━ 154s 4ms/step - DenseCho2_accuracy: 0.0890 - DenseJong2_accuracy: 0.1494 - DenseJung2_accuracy: 0.1037 - loss: 8.3554
Epoch 3/100
Helelo?
   41/41495 ━━━━━━━━━━━━━━━━━━━━ 2:41 4ms/step - DenseCho2_accuracy: 0.0232 - DenseJong2_accuracy: 0.0437 - DenseJung2_accuracy: 0.2282 - loss: 8.6443          

2024-03-18 16:31:21.561694: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
	 [[{{node IteratorGetNext}}]]
2024-03-18 16:31:21.561757: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
	 [[{{node IteratorGetNext}}]]
	 [[IteratorGetNext/_8]]
2024-03-18 16:31:21.561788: I tensorflow/core/framework/local_rendezvous.cc:422] Local rendezvous recv item cancelled. Key hash: 9191991864429175534
2024-03-18 16:31:21.561803: I tensorflow/core/framework/local_rendezvous.cc:422] Local rendezvous recv item cancelled. Key hash: 11718925914193467114
2024-03-18 16:31:21.561829: I tensorflow/core/framework/local_rendezvous.cc:422] Local rendezvous recv item cancelled. Key hash: 14274495685733646192


41495/41495 ━━━━━━━━━━━━━━━━━━━━ 155s 4ms/step - DenseCho2_accuracy: 0.0890 - DenseJong2_accuracy: 0.1494 - DenseJung2_accuracy: 0.1037 - loss: 8.3555
Epoch 4/100
Helelo?
   43/41495 ━━━━━━━━━━━━━━━━━━━━ 2:34 4ms/step - DenseCho2_accuracy: 0.0254 - DenseJong2_accuracy: 0.0472 - DenseJung2_accuracy: 0.2252 - loss: 8.6307          

2024-03-18 16:33:56.400270: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
	 [[{{node IteratorGetNext}}]]
2024-03-18 16:33:56.400313: I tensorflow/core/framework/local_rendezvous.cc:422] Local rendezvous recv item cancelled. Key hash: 11718925914193467114
2024-03-18 16:33:56.400322: I tensorflow/core/framework/local_rendezvous.cc:422] Local rendezvous recv item cancelled. Key hash: 14274495685733646192
2024-03-18 16:33:56.400388: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
	 [[{{node IteratorGetNext}}]]
	 [[IteratorGetNext/_6]]


41495/41495 ━━━━━━━━━━━━━━━━━━━━ 153s 4ms/step - DenseCho2_accuracy: 0.0890 - DenseJong2_accuracy: 0.1494 - DenseJung2_accuracy: 0.1037 - loss: 8.3556
Epoch 5/100
Helelo?
   44/41495 ━━━━━━━━━━━━━━━━━━━━ 2:33 4ms/step - DenseCho2_accuracy: 0.0264 - DenseJong2_accuracy: 0.0487 - DenseJung2_accuracy: 0.2243 - loss: 8.6238          

2024-03-18 16:36:29.836667: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
	 [[{{node IteratorGetNext}}]]
2024-03-18 16:36:29.836708: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
	 [[{{node IteratorGetNext}}]]
	 [[IteratorGetNext/_6]]
2024-03-18 16:36:29.836727: I tensorflow/core/framework/local_rendezvous.cc:422] Local rendezvous recv item cancelled. Key hash: 9191991864429175534
2024-03-18 16:36:29.836732: I tensorflow/core/framework/local_rendezvous.cc:422] Local rendezvous recv item cancelled. Key hash: 10220727970903567242
2024-03-18 16:36:29.836737: I tensorflow/core/framework/local_rendezvous.cc:422] Local rendezvous recv item cancelled. Key hash: 11718925914193467114
2024-03-18 16:36:29.836759: I tensorflow/core/framework/local_rendezvous.cc:422] Local rendezvous recv item cancelled. Key hash: 14274495685733646192


18032/41495 ━━━━━━━━━━━━━━━━━━━━ 1:31 4ms/step - DenseCho2_accuracy: 0.0889 - DenseJong2_accuracy: 0.1489 - DenseJung2_accuracy: 0.1038 - loss: 8.3570